In [1]:
!pip install playwright

In [2]:
!playwright install chromium

In [1]:
# ── STEP 1: One-time login ───────────────────────────────────────────────────
# Browser opens → fill email & password → window closes by itself when done.
# Re-run only if Step 2 says "Session expired".

import asyncio, sys, threading
from playwright.async_api import async_playwright

REPORT_URL  = "https://app.powerbi.com/groups/me/reports/ae54590b-8f16-4906-8072-ab0eae6b063b/d39ad2675706bb38711d?experience=power-bi"
PROFILE_DIR = "./powerbi_profile"

LOGIN_SELECTORS = (
    'input[type="email"], input[type="password"], '
    'input[name="loginfmt"], input[name="passwd"], '
    '#i0116, #i0118, '
    'input[placeholder*="mail" i], input[placeholder*="mot" i], '
    '.ms-TextField input'
)

async def setup_login():
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR,
            headless=False,
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()
        await page.goto(REPORT_URL)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass
        await page.wait_for_timeout(5000)

        print("⏳  Loading login page...")
        form_appeared = False
        try:
            await page.wait_for_selector(LOGIN_SELECTORS, state="visible", timeout=15000)
            form_appeared = True
        except Exception:
            pass

        if not form_appeared:
            already_in = (
                "app.powerbi.com" in page.url and
                not await page.evaluate(f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)")
            )
            if already_in:
                print("✅  Already authenticated — session saved.")
                await context.close()
                return

        print("=" * 55)
        print("🔐  Fill in your email and password in the")
        print("    browser window that just opened.")
        print("=" * 55)

        for tick in range(120):
            await page.wait_for_timeout(3000)
            try:
                url      = page.url
                has_form = await page.evaluate(
                    f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)"
                )
                if "app.powerbi.com" in url and not has_form:
                    await page.wait_for_timeout(2000)
                    has_form2 = await page.evaluate(
                        f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)"
                    )
                    if not has_form2:
                        break
            except Exception:
                pass
            if tick > 0 and tick % 10 == 0:
                print(f"   Still waiting... ({tick * 3}s)")
        else:
            print("⏱  6-minute timeout. Re-run this cell and try again.")
            await context.close()
            return

        await page.wait_for_timeout(2000)
        print("✅  Logged in! Session saved — run Step 2 now.")
        await context.close()

def run():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(setup_login())
    finally:
        loop.close()

thread = threading.Thread(target=run)
thread.start()
thread.join()

⏳  Loading login page...
✅  Already authenticated — session saved.


In [2]:
# ── STEP 2: Extract data + drill-through + send HTML email ───────────────────

import asyncio, sys, threading, smtplib, json, os, re, base64
import requests as req_lib
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from playwright.async_api import async_playwright

# ── Configuration ─────────────────────────────────────────────────────────────
REPORT_URL  = "https://app.powerbi.com/groups/me/reports/ae54590b-8f16-4906-8072-ab0eae6b063b/d39ad2675706bb38711d?experience=power-bi"
REPORT_ID   = "ae54590b-8f16-4906-8072-ab0eae6b063b"
PROFILE_DIR = "./powerbi_profile"
EMAIL_FROM  = "aalabouzid2002@gmail.com"
EMAIL_PASSWORD = "nwsj xmwh jvxz rspv"
EMAIL_TO    = ["ala-eddine.bouzid@esprit.tn"]
API_BASE    = "https://api.powerbi.com/v1.0/myorg"
XLSX_PATH   = "dépôt2026_nettoyé.xlsx"
LOGO_PATH   = "logo_laposte.jpg"

_REGIONS = [
    "ARIANA","BEJA","BEN AROUS","BIZERTE","GABES","GAFSA","JENDOUBA",
    "KAIROUAN","KASSERINE","KEBILI","KEF","MAHDIA","MANOUBA","MEDENINE",
    "MONASTIR","NABEUL","SFAX","SIDI BOUZID","SILIANA","SOUSSE",
    "TATAOUINE","TOZEUR","TUNIS","ZAGHOUAN",
]
MANUAL_SLICER_MAP = {r: "Region Depot" for r in _REGIONS}

_COMPUTED_COLS = {
    frozenset({"Agences", "Centres de distribution", "Bureaux"}): (
        "Categorie_bureau_exp",
        lambda v: "Agences" if "agence" in str(v).lower()
                  else "Centres de distribution" if "centre" in str(v).lower()
                  else "Bureaux"
    ),
}

_BLANK_TOKENS = {"(blank)", "(vide)", "(empty)", "(null)", ""}

C_NAVY   = "#0B2A6F"
C_YELLOW = "#F4C20D"
C_BG     = "#F7F9FC"
C_LIGHT  = "#EEF4FF"

_schema_cache = {}

# ── Logo helper ───────────────────────────────────────────────────────────────

def get_logo_b64():
    if os.path.exists(LOGO_PATH):
        with open(LOGO_PATH, "rb") as f:
            return "data:image/png;base64," + base64.b64encode(f.read()).decode()
    return None

# ── REST API helpers ──────────────────────────────────────────────────────────

def pbi_get(token, path):
    return req_lib.get(f"{API_BASE}{path}",
                       headers={"Authorization": f"Bearer {token}"}).json()

def get_dataset_id(token):
    for path in [f"/reports/{REPORT_ID}", f"/groups/me/reports/{REPORT_ID}"]:
        data = pbi_get(token, path)
        if "datasetId" in data:
            print(f"  ↳ Dataset ID: {data['datasetId']}")
            return data["datasetId"]
    return None

def run_dax(token, dataset_id, dax, silent=False):
    resp = req_lib.post(
        f"{API_BASE}/datasets/{dataset_id}/executeQueries",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"queries": [{"query": dax}], "serializerSettings": {"includeNulls": True}}
    )
    data = resp.json()
    if "error" in data:
        if not silent:
            print(f"  ❌ DAX error: {data.get('error', {}).get('code', '')}")
        return None
    try:
        return data["results"][0]["tables"][0].get("rows", [])
    except (KeyError, IndexError):
        return []

# ── Dynamic schema discovery ──────────────────────────────────────────────────

def find_pbi_table(token, dataset_id):
    cache = _schema_cache.setdefault(dataset_id, {})
    if "tbl" in cache:
        return cache["tbl"]
    rows = run_dax(token, dataset_id,
                   "EVALUATE SELECTCOLUMNS(INFO.TABLES(), \"n\", [Name])", silent=True)
    if rows:
        skip = {"dateautotable", "localdatetable"}
        candidates = [r.get("[n]") or r.get("n") or "" for r in rows]
        candidates = [n for n in candidates
                      if n and not n.startswith("$") and n.lower() not in skip]
        for name in candidates:
            if run_dax(token, dataset_id,
                       f"EVALUATE SELECTCOLUMNS(TOPN(1,'{name}'),\"x\","
                       f"'{name}'[MAILITM_FID])", silent=True) is not None:
                print(f"  ↳ Power BI table auto-discovered: '{name}'")
                cache["tbl"] = name
                return name
    for name in ["export", "Export", "dépôt2026_nettoyé"]:
        if run_dax(token, dataset_id, f"EVALUATE TOPN(1,'{name}')", silent=True) is not None:
            print(f"  ↳ Power BI table (fallback): '{name}'")
            cache["tbl"] = name
            return name
    return None

def get_pbi_columns(token, dataset_id, tbl_name):
    cache = _schema_cache.setdefault(dataset_id, {})
    key   = f"cols_{tbl_name}"
    if key in cache:
        return cache[key]
    rows = run_dax(token, dataset_id,
                   f"EVALUATE SELECTCOLUMNS("
                   f"FILTER(INFO.COLUMNS(), [TableName] = \"{tbl_name}\"),"
                   f"\"c\", [ExplicitName])", silent=True)
    cols = [r.get("[c]") or r.get("c") or "" for r in (rows or [])]
    cols = [c for c in cols if c]
    cache[key] = cols
    return cols

def is_dax_filterable(token, dataset_id, tbl_name, col):
    return run_dax(token, dataset_id,
                   f"EVALUATE SELECTCOLUMNS(TOPN(1,'{tbl_name}'),\"t\","
                   f"'{tbl_name}'[{col}])", silent=True) is not None

def auto_resolve_slicer_dax(token, dataset_id, tbl_name, selected_values):
    if not (token and dataset_id and tbl_name):
        return None, None
    all_cols = get_pbi_columns(token, dataset_id, tbl_name)
    if not all_cols:
        return None, None
    SKIP_KW = ('ID','FID','NUM','DATE','TIME','AMOUNT','QTY',
               'COUNT','SUM','AVG','YEAR','MONTH','WEEK')
    candidates = [c for c in all_cols
                  if not any(k in c.upper() for k in SKIP_KW)][:20]
    real_vals = [v for v in selected_values if v.lower().strip() not in _BLANK_TOKENS]
    if not real_vals:
        return None, None
    val_list = ", ".join(f'"{v}"' for v in real_vals)
    n_vals   = len(set(real_vals))
    for col in candidates:
        try:
            rows = run_dax(token, dataset_id,
                           f"EVALUATE FILTER(VALUES('{tbl_name}'[{col}]),"
                           f"'{tbl_name}'[{col}] IN {{{val_list}}})", silent=True)
            if rows is not None and len(rows) >= n_vals:
                return col, is_dax_filterable(token, dataset_id, tbl_name, col)
        except Exception:
            continue
    return None, None

# ── xlsx helpers ──────────────────────────────────────────────────────────────

def norm(s):
    return re.sub(r'[\s_\-]+', '', str(s).lower())

def has_blank_selection(vals):
    return any(v.lower().strip() in _BLANK_TOKENS for v in vals)

def xlsx_filter(df, col, selected_vals):
    real = [v for v in selected_vals if v.lower().strip() not in _BLANK_TOKENS]
    has_blank = has_blank_selection(selected_vals)
    if has_blank and real:
        return df[df[col].str.strip().isin([""] + real)]
    if has_blank:
        return df[df[col].str.strip() == ""]
    return df[df[col].isin(real)]

def load_xlsx_df():
    df = pd.read_excel(XLSX_PATH, sheet_name="export", header=2, dtype=str).fillna("")
    bureau_col = next((c for c in df.columns if norm(c) == norm("Bureau depot")), None)
    for _, (col_name, fn) in _COMPUTED_COLS.items():
        if "bureau" in col_name.lower() and bureau_col:
            df[col_name] = df[bureau_col].apply(fn)
    return df

def find_col_for_values(df, selected_values, max_unique=50):
    real_vals = [v for v in selected_values if v.lower().strip() not in _BLANK_TOKENS]
    if not real_vals:
        return None
    candidates = []
    for col in df.columns:
        unique = set(df[col].str.strip())
        if len(unique) <= max_unique and all(v in unique for v in real_vals):
            candidates.append((len(unique), col))
    if candidates:
        candidates.sort()
        return candidates[0][1]
    return None

def fix_slicer_titles(slicers, df, token=None, dataset_id=None, tbl_name=None):
    BAD = {"clear selections", "clear selection", ""}
    for s in slicers:
        raw = s["title"].lower().strip()
        if raw not in BAD and "clear" not in raw:
            if has_blank_selection(s.get("selected", [])):
                s["dax_filterable"] = False
            elif token and dataset_id and tbl_name:
                s["dax_filterable"] = is_dax_filterable(token, dataset_id, tbl_name, s["title"])
            else:
                s["dax_filterable"] = True
            continue
        # 1. Manual map
        for val in s["selected"]:
            if val in MANUAL_SLICER_MAP:
                s["title"]          = MANUAL_SLICER_MAP[val]
                s["dax_filterable"] = True
                print(f"  ↳ Slicer resolved (manual map): '{s['title']}' ← {s['selected']}")
                break
        if s["title"] and s["title"].lower() not in BAD:
            continue
        # 2. Computed DAX columns
        sel_set = frozenset(s["selected"])
        for val_set, (col_name, _) in _COMPUTED_COLS.items():
            if sel_set <= val_set:
                s["title"]          = col_name
                s["dax_filterable"] = False
                print(f"  ↳ Slicer resolved (computed DAX col): '{col_name}' ← {s['selected']}")
                break
        if s["title"] and s["title"].lower() not in BAD:
            continue
        # 3. DAX auto-discovery
        if token and dataset_id and tbl_name:
            col, filterable = auto_resolve_slicer_dax(token, dataset_id, tbl_name, s["selected"])
            if col:
                filterable = filterable and not has_blank_selection(s["selected"])
                s["title"]          = col
                s["dax_filterable"] = filterable
                mode = "DAX+xlsx" if filterable else "xlsx only"
                print(f"  ↳ Slicer resolved (DAX auto, {mode}): '{col}' ← {s['selected']}")
                continue
        # 4. xlsx auto-detect
        col = find_col_for_values(df, s["selected"])
        if col:
            s["title"]          = col
            s["dax_filterable"] = not has_blank_selection(s["selected"])
            print(f"  ↳ Slicer resolved (xlsx auto): '{col}' ← {s['selected']}")
            continue
        s["title"] = ""
        print(f"  ⚠ Slicer {s['selected']} unresolved — filter skipped")
    return slicers

def extract_measure_base(h):
    for p in ['sum of ', 'average of ', 'avg of ', 'max of ', 'min of ']:
        if h.lower().startswith(p):
            return h[len(p):]
    return None

def get_detail_cols(headers, df):
    result = []
    for h in headers:
        if 'COUNT' in h.upper() and 'MAILITM' in h.upper():
            continue
        base = extract_measure_base(h)
        if base is None:
            continue
        xlsx_col = next((c for c in df.columns if norm(c) == norm(base)), None)
        if xlsx_col:
            result.append((base, xlsx_col))
    return result

# ── Drill-through ─────────────────────────────────────────────────────────────

def build_drill_mappings(token, dataset_id, tables, table_meta, slicers, df):
    tbl_name = find_pbi_table(token, dataset_id) if token and dataset_id else None
    slicers  = fix_slicer_titles(slicers, df, token, dataset_id, tbl_name)

    fdf = df.copy()
    for s in slicers:
        if not s["title"] or not s["selected"]: continue
        col = next((c for c in fdf.columns if norm(c) == norm(s["title"])), None)
        if col:
            before = len(fdf)
            fdf = xlsx_filter(fdf, col, s["selected"])
            print(f"  ↳ xlsx filter: [{col}] = {s['selected']} → {len(fdf)} rows (was {before})")
        else:
            print(f"  ⚠ [{s['title']}] not in xlsx — skipped")

    if "MAILITM_FID" in fdf.columns:
        valid_ids = set(fdf["MAILITM_FID"].str.strip())
    else:
        valid_ids = None

    region_next_col = next((c for c in fdf.columns
                             if norm(c) in {"regionnext", norm("Region Next")}), None)
    if region_next_col:
        print(f"  ↳ Colonne Region Next : '{region_next_col}'")

    if tbl_name:
        slicer_filters = []
        for s in slicers:
            if not s["title"] or not s["selected"]: continue
            if not s.get("dax_filterable", True):
                print(f"  ↳ [{s['title']}] not DAX-filterable — post-filtering via xlsx")
                continue
            real_vals = [v for v in s["selected"] if v.lower().strip() not in _BLANK_TOKENS]
            if not real_vals:
                print(f"  ↳ [{s['title']}] blank-only — post-filtering via xlsx")
                continue
            vals, col = real_vals, s["title"]
            if len(vals) == 1:
                slicer_filters.append(f"'{tbl_name}'[{col}] = \"{vals[0]}\"")
            else:
                slicer_filters.append(
                    f"'{tbl_name}'[{col}] IN {{{', '.join(chr(34)+v+chr(34) for v in vals)}}}")
            print(f"  ↳ DAX filter: [{col}] = {vals}")

        mappings, all_ok = {}, True
        for meta, t in zip(table_meta, tables):
            if not has_mailitm_count(meta["headers"]): continue
            dim_idx = find_dim_col(meta["headers"])
            if dim_idx is None: continue
            dim_col     = meta["headers"][dim_idx]
            detail_cols = get_detail_cols(meta["headers"], fdf)
            print(f"\n  ↳ DAX query '{meta['key']}' "
                  f"(dim: [{dim_col}], detail: {[b for b,_ in detail_cols]})...")
            filters_str = (",\n    " + ",\n    ".join(slicer_filters)) if slicer_filters else ""
            dax = f"""EVALUATE
CALCULATETABLE(
    SELECTCOLUMNS(
        '{tbl_name}',
        "DimVal", '{tbl_name}'[{dim_col}],
        "ID",     '{tbl_name}'[MAILITM_FID]
    ){filters_str}
)"""
            rows = run_dax(token, dataset_id, dax)
            if rows is None:
                all_ok = False; break

            lookup = {}
            for _, xrow in fdf.iterrows():
                xid = str(xrow["MAILITM_FID"]).strip()
                if xid and xid not in lookup:
                    entry = {b: str(xrow[xc]).strip() for b, xc in detail_cols}
                    if region_next_col:
                        rn = str(xrow[region_next_col]).strip()
                        entry["Provenance"] = "Région Next vide" if not rn else "Région Dépôt"
                    lookup[xid] = entry

            flat_map, skipped, seen_per_dv = {}, 0, {}
            for row in rows:
                dv  = row.get("[DimVal]") or row.get("DimVal") or ""
                id_ = row.get("[ID]")    or row.get("ID")    or ""
                if not (dv and id_): continue
                if valid_ids is not None and id_ not in valid_ids:
                    skipped += 1; continue
                if id_ in seen_per_dv.setdefault(dv, set()): continue
                seen_per_dv[dv].add(id_)
                flat_map.setdefault(dv, []).append(
                    {"id": id_, **lookup.get(id_, {b: "" for b, _ in detail_cols})})

            total = sum(len(v) for v in flat_map.values())
            note  = f" ({skipped} post-filtered)" if skipped else ""
            print(f"    ✓ {len(flat_map)} dim vals, {total} unique IDs{note}")
            extra_cols = ["Provenance"] if region_next_col else []
            mappings[meta["key"]] = {"cols": [b for b, _ in detail_cols] + extra_cols, "data": flat_map}

        if all_ok and mappings:
            return mappings

    print("\n  ↳ Using xlsx fallback...")
    mappings = {}
    for meta, t in zip(table_meta, tables):
        if not has_mailitm_count(meta["headers"]): continue
        dim_idx = find_dim_col(meta["headers"])
        if dim_idx is None: continue
        disp = meta["headers"][dim_idx]
        col  = next((c for c in fdf.columns if norm(c) == norm(disp)), None)
        if not col:
            print(f"  ⚠ '{disp}' not found in xlsx"); continue
        detail_cols = get_detail_cols(meta["headers"], fdf)
        flat_map, seen_per_dv = {}, {}
        for _, row in fdf.iterrows():
            dv, id_ = str(row[col]).strip(), str(row["MAILITM_FID"]).strip()
            if dv and id_:
                if id_ in seen_per_dv.setdefault(dv, set()): continue
                seen_per_dv[dv].add(id_)
                entry = {"id": id_, **{b: str(row[xc]).strip() for b, xc in detail_cols}}
                if region_next_col:
                    rn = str(row[region_next_col]).strip()
                    entry["Provenance"] = "Région Next vide" if not rn else "Région Dépôt"
                flat_map.setdefault(dv, []).append(entry)
        total = sum(len(v) for v in flat_map.values())
        print(f"  ↳ xlsx '{meta['key']}': {len(flat_map)} dim vals, {total} unique IDs, "
              f"cols: {[b for b,_ in detail_cols]}")
        extra_cols = ["Provenance"] if region_next_col else []
        mappings[meta["key"]] = {"cols": [b for b, _ in detail_cols] + extra_cols, "data": flat_map}
    return mappings

# ── Playwright JS ─────────────────────────────────────────────────────────────

JS_CLASSIFY = """() => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'));
    return c.map((v, i) => {
        const isTable = !!v.querySelector('[role="columnheader"]');
        const t = v.querySelector('[data-testid="unselectable sidePaneTitle"]')
               || v.querySelector('[class*="visualTitle"] span')
               || v.querySelector('[class*="title"] span');
        return { index: i, type: isTable ? 'table' : 'other',
                 title: t ? t.textContent.trim() : '' };
    });
}"""
JS_SLICERS_GLOBAL = r"""() => {
    const groups = new Map();
    let cbs = Array.from(document.querySelectorAll('[data-testid="slicerCheckbox selected"]'));
    if (!cbs.length)
        cbs = Array.from(document.querySelectorAll('[role="checkbox"][aria-checked="true"]'));
    for (const cb of cbs) {
        const cont = cb.closest('[class*="visualContainer"]')
                  || cb.closest('[class*="visual-container"]')
                  || cb.closest('[data-testid="visual-container"]');
        if (!cont) continue;
        if (!groups.has(cont)) {
            const titleEl =
                cont.querySelector('[data-testid="unselectable sidePaneTitle"]')
             || cont.querySelector('[class*="visualTitle"] span')
             || cont.querySelector('[class*="title"] span')
             || cont.querySelector('[class*="slicerHeader"] .slicer-header-text')
             || cont.querySelector('[class*="slicerHeader"]')
             || cont.querySelector('[aria-label]');
            let title = titleEl
                ? (titleEl.getAttribute('aria-label') || titleEl.textContent || '').trim()
                : '';
            title = title.replace(/clear\s+selections?/gi, '').trim();
            groups.set(cont, { title, selected: [] });
        }
        let text = cb.getAttribute('aria-label') || '';
        if (!text) {
            const row = cb.parentElement;
            const te  = row && row.querySelector('[data-testid="slicerText"]');
            text = te ? te.textContent.trim()
                      : (row ? row.textContent.trim() : cb.textContent.trim());
        }
        if (text === '') text = '(Blank)';
        groups.get(cont).selected.push(text);
    }
    return Array.from(groups.values())
        .filter(g => g.selected.length)
        .map(g => ({ title: g.title, selected: [...new Set(g.selected)] }));
}"""
JS_HEADERS      = """(idx) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    return Array.from(c.querySelectorAll('[role="columnheader"]'))
        .map(el => el.textContent.trim()).filter(t => t && t !== 'Row Selection');
}"""
JS_VISIBLE_ROWS = """([idx, n]) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    const cells = Array.from(c.querySelectorAll('[role="gridcell"]'))
        .map(el => el.textContent.trim()).filter(t => t !== 'Select Row');
    const rows = [];
    for (let i = 0; i + n <= cells.length; i += n) rows.push(cells.slice(i, i + n));
    return rows;
}"""
JS_SCROLL       = """([idx, d]) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    let el = c.querySelector('[role="grid"]');
    if (!el || el.scrollHeight <= el.clientHeight + 5)
        el = Array.from(c.querySelectorAll('div')).find(d => {
            const s = window.getComputedStyle(d);
            return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
        });
    if (!el) return { done: true };
    el.scrollTop += d;
    return { done: el.scrollTop + el.clientHeight >= el.scrollHeight - 5 };
}"""
JS_RESET_SCROLL = """(idx) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    let el = c.querySelector('[role="grid"]');
    if (!el || el.scrollHeight <= el.clientHeight + 5)
        el = Array.from(c.querySelectorAll('div')).find(d => {
            const s = window.getComputedStyle(d);
            return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
        });
    if (el) el.scrollTop = 0;
}"""

def put_totals_last(rows):
    return [r for r in rows if all(c.strip() for c in r)] + \
           [r for r in rows if not all(c.strip() for c in r)]

def find_dim_col(headers):
    for i, h in enumerate(headers):
        if not any(kw in h.upper() for kw in ['SUM','COUNT','AVG','AVERAGE','MAX','MIN']):
            return i
    return None

def has_mailitm_count(headers):
    return any('COUNT' in h.upper() and 'MAILITM' in h.upper() for h in headers)

def is_mailitm_table(headers):
    return len(headers) == 1 and 'MAILITM' in headers[0].upper()

async def extract_all_rows(page, idx, num_cols):
    all_rows, seen = [], set()
    await page.evaluate(JS_RESET_SCROLL, idx)
    await page.wait_for_timeout(400)
    while True:
        rows = await page.evaluate(JS_VISIBLE_ROWS, [idx, num_cols])
        nf = False
        for row in rows:
            k = "|||".join(row)
            if k not in seen:
                seen.add(k); all_rows.append(row); nf = True
        result = await page.evaluate(JS_SCROLL, [idx, 120])
        await page.wait_for_timeout(300)
        if result.get("done"):
            for row in await page.evaluate(JS_VISIBLE_ROWS, [idx, num_cols]):
                k = "|||".join(row)
                if k not in seen:
                    seen.add(k); all_rows.append(row)
            break
        if not nf: break
    return put_totals_last(all_rows)

# ── HTML builders ─────────────────────────────────────────────────────────────

def _table_html_email(t, n):
    label = t["title"] or f"Tableau {n} — {' | '.join(t['headers'][:3])}"
    th = "".join(
        f'<th style="padding:9px 14px;background:{C_NAVY};color:#fff;'
        f'text-align:left;white-space:nowrap;font-size:12px;">{h}</th>'
        for h in t["headers"])
    rows_html = ""
    for i, row in enumerate(t["rows"]):
        is_tot = not all(c.strip() for c in row)
        if is_tot:
            cells = "".join(
                f'<td style="padding:7px 14px;background:#FFF8E1;font-weight:bold;'
                f'border-top:2px solid {C_YELLOW};font-size:12px;">{c}</td>' for c in row)
        else:
            bg = "#fff" if i % 2 == 0 else C_BG
            cells = "".join(
                f'<td style="padding:7px 14px;border-bottom:1px solid #E4EAF5;'
                f'background:{bg};font-size:12px;">{c}</td>' for c in row)
        rows_html += f"<tr>{cells}</tr>"
    return (
        f'<div style="margin-bottom:28px;">'
        f'<div style="background:{C_NAVY};color:#fff;padding:10px 16px;font-size:13px;'
        f'font-weight:bold;border-radius:6px 6px 0 0;">{label}</div>'
        f'<p style="color:#555;font-size:11px;margin:0;padding:6px 16px;'
        f'background:{C_LIGHT};border:1px solid #D5E1F5;border-top:none;">'
        f'{t["num_rows"]} ligne(s) &times; {t["num_cols"]} colonne(s)</p>'
        f'<div style="overflow-x:auto;border:1px solid #D5E1F5;border-top:none;'
        f'border-radius:0 0 6px 6px;">'
        f'<table style="border-collapse:collapse;width:100%;font-family:Arial,sans-serif;">'
        f'<thead><tr>{th}</tr></thead><tbody>{rows_html}</tbody></table></div></div>'
    )

def build_email_html(filename, slicers, tables, timestamp, has_logo=False, logo_b64=None):
    if slicers:
        fi_rows = "".join(
            f'<tr><td style="padding:3px 0;font-size:13px;color:#333;">'
            f'<span style="display:inline-block;background:{C_NAVY};color:{C_YELLOW};'
            f'padding:2px 10px;border-radius:3px;font-size:11px;font-weight:bold;'
            f'margin-right:8px;">{s["title"] or "Filtre"}</span>'
            f'{", ".join(v if v.lower() != "(blank)" else "(vide)" for v in s["selected"])}'
            f'</td></tr>'
            for s in slicers if s["selected"])
        filters_html = f'<table cellpadding="0" cellspacing="0" style="width:100%;">{fi_rows}</table>'
    else:
        filters_html = (f'<p style="color:#888;font-style:italic;margin:0;font-size:13px;">'
                        f'Aucun filtre actif détecté.</p>')

    tables_html = "".join(_table_html_email(t, i+1) for i, t in enumerate(tables))

    # Prefer base64 data-URI (works in all clients); fallback to CID if logo not available
    if logo_b64:
        logo_html = (f'<img src="{logo_b64}" alt="La Poste Tunisienne" '
                     f'style="height:52px;vertical-align:middle;margin-right:14px;">')
    elif has_logo:
        logo_html = (f'<img src="cid:logo_img" alt="La Poste Tunisienne" '
                     f'style="height:52px;vertical-align:middle;margin-right:14px;">')
    else:
        logo_html = ""

    date_fr = datetime.strptime(timestamp, "%Y-%m-%d %H:%M").strftime("%d/%m/%Y à %H:%M")
    logo_margin = "66px" if (logo_b64 or has_logo) else "2px"

    return f"""<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0"></head>
<body style="margin:0;padding:0;background:#EAEEF6;font-family:'Segoe UI',Arial,sans-serif;">
<table width="100%" cellpadding="0" cellspacing="0" style="background:#EAEEF6;padding:24px 0;">
<tr><td align="center">
<table width="680" cellpadding="0" cellspacing="0"
  style="background:#fff;border-radius:10px;overflow:hidden;
         box-shadow:0 4px 20px rgba(11,42,111,.15);max-width:680px;">
  <tr><td style="background:{C_NAVY};padding:0;">
    <table width="100%" cellpadding="0" cellspacing="0">
      <tr>
        <td style="padding:22px 30px 18px;vertical-align:middle;">
          {logo_html}
          <span style="color:#fff;font-size:20px;font-weight:700;letter-spacing:.5px;vertical-align:middle;">
            LA POSTE TUNISIENNE</span><br>
          <span style="color:{C_YELLOW};font-size:11px;letter-spacing:2px;text-transform:uppercase;
                        margin-left:{logo_margin};">
            Rapport Automatisé &mdash; Export International</span>
        </td>
        <td style="padding:22px 30px 18px;text-align:right;white-space:nowrap;vertical-align:middle;">
          <span style="background:{C_YELLOW};color:{C_NAVY};padding:6px 14px;
                        border-radius:20px;font-size:12px;font-weight:700;">{date_fr}</span>
        </td>
      </tr>
    </table>
    <div style="height:4px;background:{C_YELLOW};"></div>
  </td></tr>
  <tr><td style="padding:32px 36px 24px;">
    <p style="color:{C_NAVY};font-size:15px;font-weight:600;margin:0 0 16px;">Madame, Monsieur,</p>
    <p style="color:#444;font-size:14px;line-height:1.75;margin:0 0 14px;">
      Veuillez trouver ci-dessous le <strong>rapport automatisé de suivi des expéditions
      Export International</strong>, généré le <strong>{date_fr}</strong>.</p>
    <p style="color:#444;font-size:14px;line-height:1.75;margin:0 0 28px;">
      Ce rapport présente une synthèse des indicateurs (CA, poids, nombre d'envois)
      filtrés selon les critères actifs. Un <strong>fichier HTML interactif</strong> est joint
      pour accéder aux identifiants <strong>MAILITM_FID</strong> avec leurs valeurs CA et poids.</p>
    <table width="100%" cellpadding="0" cellspacing="0" style="margin-bottom:28px;">
      <tr><td style="background:{C_LIGHT};border-left:4px solid {C_NAVY};
                      padding:14px 18px;border-radius:0 6px 6px 0;">
        <div style="color:{C_NAVY};font-weight:700;font-size:12px;
                     text-transform:uppercase;letter-spacing:.5px;margin-bottom:10px;">
          Filtres appliqués</div>{filters_html}
      </td></tr>
    </table>
    <div style="margin-bottom:32px;">
      <div style="background:{C_NAVY};color:#fff;padding:10px 16px;font-size:13px;
                   font-weight:700;border-radius:6px 6px 0 0;">
        Aperçu du tableau de bord Power BI</div>
      <img src="cid:report_img" style="width:100%;display:block;border:1px solid #D5E1F5;
               border-top:none;border-radius:0 0 6px 6px;">
    </div>
    <div style="color:{C_NAVY};font-size:14px;font-weight:700;
                 border-bottom:2px solid {C_YELLOW};padding-bottom:8px;margin-bottom:22px;">
      Données détaillées par tableau</div>
    {tables_html}
    <table width="100%" cellpadding="0" cellspacing="0">
      <tr><td style="background:#FFFBE6;border:1px solid {C_YELLOW};border-radius:8px;padding:16px 20px;">
        <span style="font-size:13px;color:{C_NAVY};line-height:1.6;">
          <strong>Fichier interactif joint (.html)</strong> &mdash;
          Ouvrez la pièce jointe dans votre navigateur pour consulter les identifiants
          <strong>MAILITM_FID</strong> avec les valeurs <strong>CA</strong> et <strong>poids</strong>.
        </span>
      </td></tr>
    </table>
    <p style="color:#888;font-size:12px;line-height:1.6;margin:24px 0 0;">
      Cordialement,<br>
      <strong style="color:{C_NAVY};">Direction des Systèmes d'Information</strong><br>
      La Poste Tunisienne</p>
  </td></tr>
  <tr><td style="background:{C_NAVY};padding:0;">
    <div style="height:3px;background:{C_YELLOW};"></div>
    <table width="100%" cellpadding="0" cellspacing="0">
      <tr><td style="padding:18px 30px;text-align:center;">
        <div style="color:{C_YELLOW};font-size:13px;font-weight:700;letter-spacing:1px;margin-bottom:4px;">
          LA POSTE TUNISIENNE</div>
        <div style="color:rgba(255,255,255,.55);font-size:11px;margin-bottom:6px;">
          Direction des Systèmes d'Information &bull; Rapport généré automatiquement</div>
        <div style="color:rgba(255,255,255,.35);font-size:10px;">
          Ce message est généré automatiquement — merci de ne pas y répondre directement.</div>
      </td></tr>
    </table>
  </td></tr>
</table>
</td></tr>
</table>
</body></html>"""

def build_interactive_html(slicers, tables, drill_mappings, timestamp, logo_b64=None):
    fi = ""
    if slicers:
        items = "".join(
            f'<li><strong style="color:{C_NAVY};">{s["title"] or "Filtre"}</strong>'
            f' : {", ".join(v if v.lower() != "(blank)" else "(vide)" for v in s["selected"])}</li>'
            for s in slicers if s["selected"])
        fi = (f'<div class="fbox"><strong>Filtres appliqués :</strong>'
              f'<ul style="margin:8px 0 0;padding-left:20px;">{items}</ul></div>')

    th_html = ""
    for n, t in enumerate(tables):
        tk        = f"t{n}"
        dm        = drill_mappings.get(tk, {})
        dm_data   = dm.get("data", {}) if isinstance(dm, dict) else {}
        has_drill = bool(dm_data)
        dim_col   = find_dim_col(t["headers"])
        label     = t["title"] or f"Tableau {n+1} — {' | '.join(t['headers'][:3])}"
        hint      = (' <span class="hint">— cliquez pour voir les identifiants</span>'
                     if has_drill else '')
        th = "".join(f'<th>{h}</th>' for h in t["headers"])
        rows_html = ""
        for row in t["rows"]:
            is_tot = not all(c.strip() for c in row)
            cells  = "".join(f'<td>{c}</td>' for c in row)
            if is_tot:
                rows_html += f'<tr class="tot">{cells}</tr>'
            elif has_drill and dim_col is not None:
                dv    = row[dim_col]
                dv_js = dv.replace("\\", "\\\\").replace("'", "\\'")
                rows_html += f'<tr class="cl" onclick="drill(\'{tk}\',\'{dv_js}\')">{cells}</tr>'
            else:
                rows_html += f'<tr>{cells}</tr>'
        th_html += (f'<h3>{label}{hint}</h3>'
                    f'<p class="meta">{t["num_rows"]} lignes &times; {t["num_cols"]} colonnes</p>'
                    f'<div class="tw"><table id="{tk}"><thead><tr>{th}</tr></thead>'
                    f'<tbody>{rows_html}</tbody></table></div>')

    js_data  = json.dumps(drill_mappings, ensure_ascii=False)
    date_fr  = datetime.strptime(timestamp, "%Y-%m-%d %H:%M").strftime("%d/%m/%Y à %H:%M")
    logo_tag = (f'<img src="{logo_b64}" alt="La Poste Tunisienne" '
                f'style="height:48px;margin-right:14px;vertical-align:middle;">'
                if logo_b64 else "")

    html = f"""<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8">
<title>Rapport Export International — La Poste Tunisienne</title>
<style>
*{{box-sizing:border-box}}
body{{font-family:'Segoe UI',Arial,sans-serif;margin:0;padding:0;background:{C_BG};color:#333}}
.wrapper{{max-width:1400px;margin:auto;padding:24px}}
.page-header{{background:{C_NAVY};border-radius:10px 10px 0 0;overflow:hidden}}
.page-header .top{{display:flex;align-items:center;justify-content:space-between;padding:20px 28px 16px}}
.page-header .brand{{color:#fff;font-size:22px;font-weight:700;letter-spacing:.5px}}
.page-header .sub{{color:{C_YELLOW};font-size:11px;letter-spacing:2px;text-transform:uppercase;margin-top:4px}}
.page-header .badge{{background:{C_YELLOW};color:{C_NAVY};padding:6px 16px;border-radius:20px;font-size:12px;font-weight:700;white-space:nowrap}}
.accent-bar{{height:4px;background:{C_YELLOW}}}
.content{{background:#fff;border-radius:0 0 10px 10px;padding:28px;box-shadow:0 4px 20px rgba(11,42,111,.1);margin-bottom:24px}}
h3{{color:{C_NAVY};border-left:4px solid {C_YELLOW};padding-left:12px;margin-top:32px;margin-bottom:6px}}
.hint{{color:{C_YELLOW};font-size:11px;font-weight:normal;font-style:italic}}
.meta{{color:#666;font-size:12px;margin:2px 0 8px}}.tw{{overflow-x:auto}}
table{{border-collapse:collapse;width:100%;font-size:13px;margin-bottom:10px;background:white;border-radius:6px;box-shadow:0 1px 6px rgba(11,42,111,.08)}}
th{{background:{C_NAVY};color:white;padding:9px 14px;text-align:left;white-space:nowrap}}
td{{padding:7px 14px;border-bottom:1px solid #E4EAF5}}
tr.cl{{cursor:pointer}}tr.cl:hover td{{background:#FFF8E1!important}}
tr.sel td{{background:#FFF3CD!important;font-weight:bold}}
tr:nth-child(even) td{{background:{C_BG}}}
tr.tot td{{background:#FFF8E1;font-weight:bold;border-top:2px solid {C_YELLOW}}}
.fbox{{background:{C_LIGHT};border-left:4px solid {C_NAVY};padding:14px 18px;margin-bottom:24px;border-radius:0 6px 6px 0}}
#panel{{display:none;position:fixed;bottom:0;left:0;right:0;background:white;
        border-top:4px solid {C_YELLOW};padding:16px 28px;
        box-shadow:0 -4px 24px rgba(11,42,111,.2);max-height:50vh;overflow-y:auto;z-index:999}}
#panel h3{{margin:0 0 4px;border-left:4px solid {C_YELLOW};padding-left:10px;font-size:15px}}
#xbtn{{float:right;cursor:pointer;font-size:24px;color:#999;background:none;border:none;line-height:1}}
.tag{{display:inline-block;background:{C_NAVY};color:{C_YELLOW};padding:3px 12px;border-radius:12px;font-size:12px;margin-left:6px;font-weight:600}}
#ptable{{width:auto;min-width:420px;box-shadow:none}}
#ptable th{{background:#1A3D8A;font-size:12px;padding:7px 12px}}
#ptable td{{font-size:12px;padding:6px 12px}}
.no-data{{color:#999;font-style:italic}}
.badge-depot{{display:inline-block;background:{C_NAVY};color:#fff;padding:2px 8px;border-radius:3px;font-size:11px;font-weight:600;white-space:nowrap}}
.badge-next{{display:inline-block;background:{C_YELLOW};color:{C_NAVY};padding:2px 8px;border-radius:3px;font-size:11px;font-weight:600;white-space:nowrap}}
footer{{text-align:center;padding:16px;background:{C_NAVY};border-radius:10px;
        color:rgba(255,255,255,.5);font-size:11px;margin-top:4px}}
footer span{{color:{C_YELLOW};font-weight:700}}
</style></head><body>
<div class="wrapper">
  <div class="page-header">
    <div class="top">
      <div style="display:flex;align-items:center;">
        {logo_tag}
        <div>
          <div class="brand">LA POSTE TUNISIENNE</div>
          <div class="sub">Rapport Export International &mdash; Interactif</div>
        </div>
      </div>
      <div class="badge">{date_fr}</div>
    </div>
    <div class="accent-bar"></div>
  </div>
  <div class="content">FILTER_INFOTABLES_HTML</div>
  <footer><span>LA POSTE TUNISIENNE</span> &bull; Direction des Systèmes d'Information &bull; Rapport généré automatiquement</footer>
</div>
<div id="panel">
  <button id="xbtn" onclick="closeP()">&#x2715;</button>
  <h3>Identifiants &mdash; <span id="lbl"></span></h3>
  <p id="cnt" style="color:#666;font-size:12px;margin:4px 0 10px"></p>
  <div style="overflow-x:auto">
    <table id="ptable"><thead id="phead"></thead><tbody id="pbody"></tbody></table>
  </div>
</div>
<script>
const D=DRILL_DATA;
function drill(tk,dv){{
  const tm=D[tk]||{{}};const cols=tm.cols||[];const td=tm.data||tm;
  let rows=td[dv];
  if(rows===undefined){{
    const dvL=dv.trim().toLowerCase();
    const k=Object.keys(td).find(k=>k.trim().toLowerCase()===dvL);
    rows=k!==undefined?td[k]:[];
  }}
  rows=Array.isArray(rows)?rows:[];
  document.getElementById('lbl').innerHTML='<span class="tag">'+dv+'</span>';
  document.getElementById('cnt').textContent=rows.length+' identifiant(s) unique(s)';
  document.getElementById('phead').innerHTML='<tr><th>MAILITM_FID</th>'+
    cols.map(c=>'<th>'+c+'</th>').join('')+'</tr>';
  if(rows.length){{
    document.getElementById('pbody').innerHTML=rows.map(r=>{{
      const id=typeof r==='string'?r:(r.id||'');
      const extras=cols.map(c=>{{
        if(c==='Provenance'){{
          const v=r[c]||'';
          const cls=v==='Région Next vide'?'badge-next':'badge-depot';
          return '<td><span class="'+cls+'">'+v+'</span></td>';
        }}
        return '<td>'+(r[c]!==undefined?r[c]:'')+'</td>';
      }}).join('');
      return '<tr><td>'+id+'</td>'+extras+'</tr>';
    }}).join('');
  }} else {{
    document.getElementById('pbody').innerHTML=
      '<tr><td class="no-data" colspan="'+(cols.length+1)+'">Aucun identifiant.</td></tr>';
  }}
  document.getElementById('panel').style.display='block';
  document.querySelectorAll('.cl').forEach(r=>r.classList.remove('sel'));
  document.querySelectorAll('#'+tk+' .cl').forEach(r=>{{
    if(Array.from(r.querySelectorAll('td')).some(c=>c.textContent===dv))r.classList.add('sel');
  }});
  document.getElementById('panel').scrollIntoView({{behavior:'smooth',block:'end'}});
}}
function closeP(){{
  document.getElementById('panel').style.display='none';
  document.querySelectorAll('.cl').forEach(r=>r.classList.remove('sel'));
}}
</script></body></html>"""
    return (html.replace("FILTER_INFO", fi)
                .replace("TABLES_HTML", th_html)
                .replace("DRILL_DATA", js_data))

# ── Main ──────────────────────────────────────────────────────────────────────

async def run_all():
    token_holder = []
    _schema_cache.clear()

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080}, device_scale_factor=2,
        )
        page = await context.new_page()

        def on_request(request):
            auth = request.headers.get("authorization", "")
            if (auth.startswith("Bearer ") and not token_holder and
                    any(d in request.url for d in ["powerbi.com", "analysis.windows.net"])):
                token_holder.append(auth[7:])

        page.on("request", on_request)
        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception: pass

        if "app.powerbi.com" not in page.url:
            print("❌ Session expired — re-run Step 1.")
            await context.close(); return

        await page.wait_for_timeout(8000)

        bearer_token = token_holder[0] if token_holder else None
        print(f"  {'✅ Bearer token captured' if bearer_token else '❌ Bearer token NOT found'}")

        slicers = await page.evaluate(JS_SLICERS_GLOBAL)
        print(f"  ↳ {len(slicers)} slicer(s): {[(s['title'], s['selected']) for s in slicers]}")

        visuals = await page.evaluate(JS_CLASSIFY)
        tables, table_meta = [], []
        for v in visuals:
            if v["type"] != "table": continue
            headers = await page.evaluate(JS_HEADERS, v["index"])
            if not headers or is_mailitm_table(headers): continue
            label = v["title"] or f"Tableau — {' | '.join(headers[:3])}"
            print(f"  ↳ Extracting '{label}'...")
            rows = await extract_all_rows(page, v["index"], len(headers))
            tkey = f"t{len(tables)}"
            tables.append({"title": v["title"], "headers": headers,
                            "rows": rows, "num_rows": len(rows), "num_cols": len(headers)})
            table_meta.append({"key": tkey, "idx": v["index"], "headers": headers})
            print(f"    ✓ {len(rows)} rows")

        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        await page.screenshot(path=filename, full_page=False)
        await context.close()

    print("\n── Building drill-through mappings ─────────────")
    try:
        df = load_xlsx_df()
    except Exception as e:
        print(f"  ⚠ Could not load xlsx: {e}")
        df = pd.DataFrame()

    dataset_id = get_dataset_id(bearer_token) if bearer_token else None
    drill_mappings = build_drill_mappings(
        bearer_token, dataset_id, tables, table_meta, slicers, df)

    for k, v in drill_mappings.items():
        cols = v.get("cols", []) if isinstance(v, dict) else []
        data = v.get("data", {}) if isinstance(v, dict) else {}
        print(f"  [{k}] {len(data)} dim vals | panel cols: {cols}")

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    has_logo  = os.path.exists(LOGO_PATH)
    logo_b64  = get_logo_b64()
    print(f"\n✅ Screenshot : {filename}")
    print(f"  Logo : {'✅ base64 (email + HTML)' if logo_b64 else '⚠ logo_laposte.jpg introuvable'}")

    html_filename = filename.replace(".png", "_interactif.html")
    with open(html_filename, "w", encoding="utf-8") as f:
        f.write(build_interactive_html(slicers, tables, drill_mappings, timestamp, logo_b64))
    print(f"✅ HTML interactif : {html_filename}")

    # Logo embedded as base64 in the email body — no CID attachment needed
    email_html = build_email_html(filename, slicers, tables, timestamp, has_logo, logo_b64)
    msg = MIMEMultipart("related")
    msg["From"]    = EMAIL_FROM
    msg["To"]      = ", ".join(EMAIL_TO)
    msg["Subject"] = (f"[La Poste Tunisienne] Rapport Export International — "
                      f"{datetime.strptime(timestamp, '%Y-%m-%d %H:%M').strftime('%d/%m/%Y')}")
    msg.attach(MIMEText(email_html, "html", "utf-8"))

    with open(filename, "rb") as f:
        img = MIMEImage(f.read())
    img.add_header("Content-ID", "<report_img>")
    img.add_header("Content-Disposition", "inline", filename=filename)
    msg.attach(img)

    with open(html_filename, "rb") as f:
        att = MIMEBase("text", "html")
        att.set_payload(f.read())
    encoders.encode_base64(att)
    att.add_header("Content-Disposition", "attachment",
                   filename=os.path.basename(html_filename))
    msg.attach(att)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_FROM, EMAIL_PASSWORD)
        server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())
    print(f"📧 Email envoyé → {EMAIL_TO}")

def run():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(run_all())
    finally:
        loop.close()

thread = threading.Thread(target=run)
thread.start()
thread.join()

  ✅ Bearer token captured
  ↳ 1 slicer(s): [('', ['Agences'])]
  ↳ Extracting 'Tableau — Sum of CA | Bureau depot | Count of MAILITM_FID'...
    ✓ 26 rows

── Building drill-through mappings ─────────────
  ↳ Dataset ID: bd4913fc-b016-4e8a-969c-a738ccd18594
  ↳ Power BI table (fallback): 'export'
  ↳ Slicer resolved (computed DAX col): 'Categorie_bureau_exp' ← ['Agences']
  ↳ xlsx filter: [Categorie_bureau_exp] = ['Agences'] → 9998 rows (was 16417)
  ↳ Colonne Region Next : 'Region Next'
  ↳ [Categorie_bureau_exp] not DAX-filterable — post-filtering via xlsx

  ↳ DAX query 't0' (dim: [Bureau depot], detail: ['CA'])...
    ✓ 29 dim vals, 9998 unique IDs (6419 post-filtered)
  [t0] 29 dim vals | panel cols: ['CA', 'Provenance']

✅ Screenshot : report_20260608_092404.png
  Logo : ✅ base64 (email + HTML)
✅ HTML interactif : report_20260608_092404_interactif.html
📧 Email envoyé → ['ala-eddine.bouzid@esprit.tn']


In [ ]:
# ── STEP 3: Email automatique par région (Catégorie = Agences) ────────────────
# Prérequis : avoir exécuté Step 2 au moins une fois (fonctions chargées en mémoire).
# Le slicer "Catégorie bureau = Agences" DOIT être déjà coché dans Power BI.
# Ce script change uniquement le slicer "Region Depot" pour chaque région.

# ╔══════════════════════════════════════════════════════╗
# ║  EMAILS PAR RÉGION — REMPLISSEZ CE DICTIONNAIRE     ║
# ║  Laissez "" pour utiliser l'adresse par défaut      ║
# ╚══════════════════════════════════════════════════════╝
REGION_EMAILS = {
    "ARIANA":      "",   # ex: "responsable.ariana@laposte.tn"
    "BEJA":        "",
    "BEN AROUS":   "",
    "BIZERTE":     "",
    "GABES":       "",
    "GAFSA":       "",
    "JENDOUBA":    "",
    "KAIROUAN":    "",
    "KASSERINE":   "",
    "KEBILI":      "",
    "KEF":         "",
    "MAHDIA":      "",
    "MANOUBA":     "",
    "MEDENINE":    "",
    "MONASTIR":    "",
    "NABEUL":      "",
    "SFAX":        "",
    "SIDI BOUZID": "",
    "SILIANA":     "",
    "SOUSSE":      "",
    "TATAOUINE":   "",
    "TOZEUR":      "",
    "TUNIS":       "",
    "ZAGHOUAN":    "",
}
DEFAULT_EMAIL  = "ala-eddine.bouzid@esprit.tn"   # adresse si case vide
FIXED_CATEGORY = "Agences"                        # catégorie fixée pour toutes les régions
# ─────────────────────────────────────────────────────────────────────────────

# ── JS pour interagir avec les slicers ────────────────────────────────────────

_REGION_NAMES_UPPER = [r.upper() for r in _REGIONS]

JS_FIND_REGION_SLICER = """(regionNames) => {
    const cs = Array.from(document.querySelectorAll('[data-testid="visual-container"]'));
    for (let i = 0; i < cs.length; i++) {
        const texts = Array.from(cs[i].querySelectorAll('[data-testid="slicerText"]'))
                          .map(el => el.textContent.trim().toUpperCase());
        if (texts.filter(t => regionNames.includes(t)).length >= 3) return i;
    }
    return -1;
}"""

JS_CLEAR_SLICER = """(idx) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return false;
    const btn = c.querySelector('[aria-label="Clear selections"]')
             || c.querySelector('[aria-label="Effacer les sélections"]')
             || c.querySelector('[title="Clear selections"]')
             || Array.from(c.querySelectorAll('button,[role="button"]')).find(b =>
                    (b.getAttribute('aria-label')||b.title||'').toLowerCase().includes('clear') ||
                    (b.getAttribute('aria-label')||b.title||'').toLowerCase().includes('effacer'));
    if (btn) { btn.click(); return true; }
    return false;
}"""

JS_SLICER_SCROLL_TOP = """(idx) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return;
    const el = Array.from(c.querySelectorAll('div')).find(d => {
        const s = window.getComputedStyle(d);
        return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
    });
    if (el) el.scrollTop = 0;
}"""

JS_TRY_SELECT_IN_SLICER = """([idx, value]) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return false;
    for (const item of c.querySelectorAll('[data-testid="slicerText"]')) {
        if (item.textContent.trim().toUpperCase() === value.toUpperCase()) {
            const row = item.closest('[role="option"]')
                     || item.closest('[class*="row"]')
                     || item.parentElement;
            if (row) { row.click(); return true; }
            item.click(); return true;
        }
    }
    return false;
}"""

JS_SCROLL_SLICER = """([idx, amount]) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return { done: true };
    const el = Array.from(c.querySelectorAll('div')).find(d => {
        const s = window.getComputedStyle(d);
        return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
    });
    if (!el) return { done: true };
    el.scrollTop += amount;
    return { done: el.scrollTop + el.clientHeight >= el.scrollHeight - 5 };
}"""

# ── Fonctions auxiliaires ─────────────────────────────────────────────────────

async def set_region_slicer(page, slicer_idx, region):
    """Clear region slicer then select target region. Returns True on success."""
    # 1. Clear
    cleared = await page.evaluate(JS_CLEAR_SLICER, slicer_idx)
    if not cleared:
        print(f"    ⚠ Clear slicer failed — trying anyway")
    await page.wait_for_timeout(700)

    # 2. Reset scroll
    await page.evaluate(JS_SLICER_SCROLL_TOP, slicer_idx)
    await page.wait_for_timeout(200)

    # 3. Select value (scroll down if needed)
    for _ in range(40):
        if await page.evaluate(JS_TRY_SELECT_IN_SLICER, [slicer_idx, region]):
            return True
        r = await page.evaluate(JS_SCROLL_SLICER, [slicer_idx, 60])
        await page.wait_for_timeout(80)
        if r.get("done"):
            break

    # One last scroll-to-top + retry
    await page.evaluate(JS_SLICER_SCROLL_TOP, slicer_idx)
    await page.wait_for_timeout(200)
    return await page.evaluate(JS_TRY_SELECT_IN_SLICER, [slicer_idx, region])

def send_region_email(filename, html_filename, slicers, tables,
                      timestamp, recipient, region, logo_b64, has_logo):
    """Build and send one email for a region."""
    email_html = build_email_html(filename, slicers, tables, timestamp, has_logo, logo_b64)
    msg = MIMEMultipart("related")
    msg["From"]    = EMAIL_FROM
    msg["To"]      = recipient
    msg["Subject"] = (
        f"[La Poste Tunisienne] Rapport Export International — "
        f"{region} — "
        f"{datetime.strptime(timestamp, '%Y-%m-%d %H:%M').strftime('%d/%m/%Y')}"
    )
    msg.attach(MIMEText(email_html, "html", "utf-8"))

    with open(filename, "rb") as f:
        img = MIMEImage(f.read())
    img.add_header("Content-ID", "<report_img>")
    img.add_header("Content-Disposition", "inline", filename=filename)
    msg.attach(img)

    with open(html_filename, "rb") as f:
        att = MIMEBase("text", "html")
        att.set_payload(f.read())
    encoders.encode_base64(att)
    att.add_header("Content-Disposition", "attachment",
                   filename=os.path.basename(html_filename))
    msg.attach(att)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_FROM, EMAIL_PASSWORD)
        server.sendmail(EMAIL_FROM, [recipient], msg.as_string())

# ── Main ──────────────────────────────────────────────────────────────────────

async def run_all_regions():
    has_logo = os.path.exists(LOGO_PATH)
    logo_b64 = get_logo_b64()

    try:
        df = load_xlsx_df()
        print(f"✅ xlsx chargé : {len(df)} lignes")
    except Exception as e:
        print(f"❌ Impossible de charger le xlsx : {e}"); return

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080}, device_scale_factor=2,
        )
        page = await context.new_page()

        token_holder = []
        def on_request(request):
            auth = request.headers.get("authorization", "")
            if auth.startswith("Bearer ") and not token_holder:
                token_holder.append(auth[7:])
        page.on("request", on_request)

        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass

        if "app.powerbi.com" not in page.url:
            print("❌ Session expirée — relancez Step 1.")
            await context.close(); return

        await page.wait_for_timeout(8000)
        print(f"  {'✅ Bearer token capturé' if token_holder else '⚠ Bearer token absent'}")

        # Vérifier que Agences est sélectionné
        current_slicers = await page.evaluate(JS_SLICERS_GLOBAL)
        has_agences = any(
            FIXED_CATEGORY in s.get("selected", []) for s in current_slicers
        )
        if not has_agences:
            print(f"❌ '{FIXED_CATEGORY}' n'est pas coché dans Power BI.")
            print(f"   → Sélectionnez-le manuellement dans le rapport, puis relancez Step 3.")
            await context.close(); return
        print(f"  ✅ Filtre '{FIXED_CATEGORY}' confirmé")

        # Trouver l'index du slicer Region Depot
        slicer_idx = await page.evaluate(JS_FIND_REGION_SLICER, _REGION_NAMES_UPPER)
        if slicer_idx < 0:
            print("❌ Slicer Region Depot introuvable dans la page.")
            await context.close(); return
        print(f"  ✅ Slicer Region Depot trouvé (container #{slicer_idx})")

        success, failed = 0, []

        for region in _REGIONS:
            recipient = REGION_EMAILS.get(region, "").strip() or DEFAULT_EMAIL
            print(f"\n── {region} → {recipient} ──")

            # Changer le slicer
            ok = await set_region_slicer(page, slicer_idx, region)
            if not ok:
                print(f"  ❌ Région '{region}' non trouvée dans le slicer — ignorée")
                failed.append(region); continue
            print(f"  ✅ Slicer mis à jour → {region}")

            # Attendre le rafraîchissement Power BI
            await page.wait_for_timeout(5000)

            # Screenshot
            filename = f"report_{region}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
            await page.screenshot(path=filename, full_page=False)

            # Extraire les tableaux
            visuals = await page.evaluate(JS_CLASSIFY)
            tables, table_meta = [], []
            for v in visuals:
                if v["type"] != "table": continue
                headers = await page.evaluate(JS_HEADERS, v["index"])
                if not headers or is_mailitm_table(headers): continue
                rows = await extract_all_rows(page, v["index"], len(headers))
                tkey = f"t{len(tables)}"
                tables.append({"title": v["title"], "headers": headers,
                                "rows": rows, "num_rows": len(rows), "num_cols": len(headers)})
                table_meta.append({"key": tkey, "idx": v["index"], "headers": headers})
            print(f"  ↳ {len(tables)} tableau(x) extrait(s)")

            # Slicers connus pour cet email
            slicers_email = [
                {"title": "Categorie_bureau_exp", "selected": [FIXED_CATEGORY],
                 "dax_filterable": False},
                {"title": "Region Depot",         "selected": [region],
                 "dax_filterable": True},
            ]

            # Drill-through via xlsx uniquement (rapide, sans appels API par région)
            drill_mappings = build_drill_mappings(
                None, None, tables, table_meta, slicers_email, df)

            timestamp    = datetime.now().strftime("%Y-%m-%d %H:%M")
            html_filename = filename.replace(".png", "_interactif.html")
            with open(html_filename, "w", encoding="utf-8") as f:
                f.write(build_interactive_html(
                    slicers_email, tables, drill_mappings, timestamp, logo_b64))

            try:
                send_region_email(filename, html_filename, slicers_email,
                                  tables, timestamp, recipient, region, logo_b64, has_logo)
                print(f"  📧 Email envoyé → {recipient}")
                success += 1
            except Exception as e:
                print(f"  ❌ Échec envoi : {e}")
                failed.append(region)

        await context.close()

    print(f"\n{'='*50}")
    print(f"✅ {success}/{len(_REGIONS)} emails envoyés")
    if failed:
        print(f"❌ Échecs : {failed}")

def run_regions():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(run_all_regions())
    finally:
        loop.close()

thread = threading.Thread(target=run_regions)
thread.start()
thread.join()

In [5]:
# ── DIAGNOSTIC: test which column names exist in the Power BI model ───────────

import asyncio, sys, threading
import requests as req_lib
from playwright.async_api import async_playwright

REPORT_URL  = "https://app.powerbi.com/groups/me/reports/ae54590b-8f16-4906-8072-ab0eae6b063b/d39ad2675706bb38711d?experience=power-bi"
REPORT_ID   = "ae54590b-8f16-4906-8072-ab0eae6b063b"
PROFILE_DIR = "./powerbi_profile"
API_BASE    = "https://api.powerbi.com/v1.0/myorg"

CANDIDATES = [
    "region_depot", "Region Depot", "Region_Depot",
    "categorie_bureau_exp", "Categorie_bureau_exp",
    "Pays Destination", "Pays_Destination",
    "MAILITM_FID", "Bureau depot", "Bureau_depot",
    "Exp_cité", "CA", "CRBT",
]

async def diagnose():
    token_holder = []
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()
        def on_request(request):
            auth = request.headers.get("authorization", "")
            if auth.startswith("Bearer ") and not token_holder:
                token_holder.append(auth[7:])
        page.on("request", on_request)
        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass
        await page.wait_for_timeout(9000)
        await context.close()

    token = token_holder[0] if token_holder else None
    print(f"Token captured: {'✅ yes' if token else '❌ NO'}")
    if not token:
        return

    dataset_id = None
    for path in [f"/reports/{REPORT_ID}", f"/groups/me/reports/{REPORT_ID}"]:
        data = req_lib.get(f"{API_BASE}{path}",
                           headers={"Authorization": f"Bearer {token}"}).json()
        if "datasetId" in data:
            dataset_id = data["datasetId"]
            print(f"Dataset ID: {dataset_id}\n")
            break
    if not dataset_id:
        print("❌ Could not get dataset ID")
        return

    print("Column name test in 'export' table:")
    print("-" * 45)
    for col in CANDIDATES:
        dax = f"EVALUATE SELECTCOLUMNS(TOPN(1, 'export'), \"t\", 'export'[{col}])"
        resp = req_lib.post(
            f"{API_BASE}/datasets/{dataset_id}/executeQueries",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json={"queries": [{"query": dax}], "serializerSettings": {"includeNulls": True}}
        )
        ok = "error" not in resp.json()
        print(f"  [{'✓' if ok else '✗'}]  {col}")

def run():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(diagnose())
    finally:
        loop.close()

thread = threading.Thread(target=run)
thread.start()
thread.join()

Token captured: ✅ yes
Dataset ID: bd4913fc-b016-4e8a-969c-a738ccd18594

Column name test in 'export' table:
---------------------------------------------
  [✗]  region_depot
  [✓]  Region Depot
  [✗]  Region_Depot
  [✗]  categorie_bureau_exp
  [✗]  Categorie_bureau_exp
  [✓]  Pays Destination
  [✗]  Pays_Destination
  [✓]  MAILITM_FID
  [✓]  Bureau depot
  [✗]  Bureau_depot
  [✓]  Exp_cité
  [✗]  CA
  [✓]  CRBT


KeyboardInterrupt: 